# 17 — Hex-Pixel Detectors (PIXIRAD-style honeycomb)

Most lab/synchrotron detectors are square-pixel Cartesian (PE, GE, Varex). A few — PIXIRAD, Hexitec, some pixel-array tracking detectors — use a **hexagonal honeycomb** lattice where rows alternate in column offset. v2's `pixel_to_REta` supports this natively via the `lattice='hex_offset_y'` mode.

The key parameter is the **apothem** (half the column pitch, in µm). For a regular honeycomb:
* row pitch in Z = `a · √3`
* column pitch in Y = `2a`
* odd-Z rows shifted by `+a` in Y

Apothem is **refinable** — it's the single intrinsic scale parameter that replaces `pxY` / `pxZ` for hex grids.

This notebook:

1. Shows the pixel-to-physical mapping for both lattices side by side.
2. Builds a synthetic CeO₂ hex-pixel image.
3. Calibrates it with `lattice='hex_offset_y'`.
4. Demonstrates `LatticeOrientation` for in-plane rotation of the hex axes relative to the detector frame.

In [ ]:
import os; os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
import math, numpy as np, torch, matplotlib.pyplot as plt

## 1. Pixel-to-physical mapping comparison

In [ ]:
from midas_calibrate_v2.forward.lattice import lattice_to_phys

# A 4×4 grid of pixel indices, with BC at (1.5, 1.5).
Y_pix, Z_pix = np.meshgrid(np.arange(4), np.arange(4))
Y = torch.as_tensor(Y_pix.ravel(), dtype=torch.float64)
Z = torch.as_tensor(Z_pix.ravel(), dtype=torch.float64)
BC_y, BC_z = torch.tensor(1.5, dtype=torch.float64), torch.tensor(1.5, dtype=torch.float64)

# Cartesian: pxY=pxZ=150 µm
Yc_c, Zc_c = lattice_to_phys(Y, Z, lattice='cartesian',
                              BC_y=BC_y, BC_z=BC_z,
                              pxY=torch.tensor(150.0), pxZ=torch.tensor(150.0))

# Hex: apothem=75 µm (so column pitch = 150 µm, row pitch = ~130 µm)
Yc_h, Zc_h = lattice_to_phys(Y, Z, lattice='hex_offset_y',
                              BC_y=BC_y, BC_z=BC_z,
                              apothem=torch.tensor(75.0))

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].scatter(Yc_c.numpy(), Zc_c.numpy(), s=200, c='steelblue', edgecolor='k')
axes[0].set_title('cartesian'); axes[0].set_aspect('equal'); axes[0].grid()
axes[1].scatter(Yc_h.numpy(), Zc_h.numpy(), s=200, c='coral', edgecolor='k')
axes[1].set_title('hex_offset_y (alternating row shift)')
axes[1].set_aspect('equal'); axes[1].grid()
for a in axes: a.set_xlabel('Y (µm)'); a.set_ylabel('Z (µm)')
plt.tight_layout(); plt.show()

## 2. Synthetic hex-pixel CeO₂ image

Use the forward model to *generate* expected ring positions on a hex grid, then add Gaussian-profile rings. This is the cleanest way to verify the calibration round-trip without needing a real PIXIRAD dataset.

In [ ]:
from midas_hkls import SpaceGroup, Lattice, generate_hkls
from midas_calibrate_v2.forward.geometry import pixel_to_REta
from midas_calibrate_v2.forward.distortion import build_p_coeffs
import math

NY = NZ = 512
TRUE = dict(Lsd=300_000.0, BC_y=256.0, BC_z=256.0, apothem=75.0,
             ty=0.05, tz=-0.03, wavelength=0.184139)
p_coeffs = torch.zeros(15, dtype=torch.float64)

# Map every pixel through the forward model -> (R_px, η_deg)
yy, zz = np.meshgrid(np.arange(NY), np.arange(NZ))
Y = torch.as_tensor(yy.ravel(), dtype=torch.float64)
Z = torch.as_tensor(zz.ravel(), dtype=torch.float64)
out = pixel_to_REta(
    Y, Z,
    Lsd=torch.tensor(TRUE['Lsd'], dtype=torch.float64),
    BC_y=torch.tensor(TRUE['BC_y'], dtype=torch.float64),
    BC_z=torch.tensor(TRUE['BC_z'], dtype=torch.float64),
    tx=torch.tensor(0.0, dtype=torch.float64),
    ty=torch.tensor(TRUE['ty'], dtype=torch.float64),
    tz=torch.tensor(TRUE['tz'], dtype=torch.float64),
    p_coeffs=p_coeffs, parallax=torch.tensor(0.0, dtype=torch.float64),
    pxY=torch.tensor(2.0*TRUE['apothem'], dtype=torch.float64),
    pxZ=torch.tensor(TRUE['apothem'] * math.sqrt(3), dtype=torch.float64),
    rho_d=torch.tensor(300.0, dtype=torch.float64),
    lattice='hex_offset_y',
    apothem=torch.tensor(TRUE['apothem'], dtype=torch.float64),
)
R_px = out.R_px.numpy().reshape(NZ, NY)

# CeO₂ ring radii via midas_hkls (proper SG 225 fcc extinction).
refs = generate_hkls(
    SpaceGroup.from_number(225),
    Lattice(a=5.4116, b=5.4116, c=5.4116, alpha=90.0, beta=90.0, gamma=90.0),
    wavelength_A=TRUE['wavelength'], two_theta_max_deg=10.0,
)
rings_R_px = [TRUE['Lsd'] * math.tan(math.radians(r.two_theta_deg))
              / (2*TRUE['apothem']) for r in refs[:5]]

img = np.full((NZ, NY), 50.0, dtype=np.float32)
for r0 in rings_R_px:
    img += 1000.0 * np.exp(-0.5 * ((R_px - r0) / 1.0) ** 2)
img += np.random.default_rng(0).normal(0, 5, size=img.shape).astype(np.float32)
print(f'synthetic image shape={img.shape} expected rings @ R={[round(r,1) for r in rings_R_px]}')


## 3. Calibrate via `autocalibrate` with a hex spec

The one-shot `calibrate()` is cartesian-only for now. For hex pixels, build the spec manually and call `autocalibrate` directly so the lattice mode propagates.

In [ ]:
from midas_calibrate.params import CalibrationParams
from midas_calibrate_v2.compat.from_v1 import spec_from_v1_params
from midas_calibrate_v2.parameters.parameter import Parameter
from midas_calibrate_v2.pipelines.single import autocalibrate

v1 = CalibrationParams(
    NrPixelsY=NY, NrPixelsZ=NZ,
    pxY=2*TRUE['apothem'], pxZ=TRUE['apothem']*math.sqrt(3),
    Lsd=TRUE['Lsd']*1.01, BC_y=TRUE['BC_y']+1.5, BC_z=TRUE['BC_z']-2.0,
    tx=0.0, ty=0.0, tz=0.0,
    Wavelength=TRUE['wavelength'], SpaceGroup=225,
    LatticeConstant=(5.4116, 5.4116, 5.4116, 90.0, 90.0, 90.0),
    MaxRingRad=300.0, MinRingRad=20.0,
    Refine={'Lsd': True, 'BC': True, 'ty': True, 'tz': True,
             'Wavelength': False, 'Parallax': False,
             **{f'p{i}': False for i in range(15)}},
    Device='cpu', Dtype='fp64',
)
spec = spec_from_v1_params(v1)
spec.lattice = 'hex_offset_y'
spec.add(Parameter('Apothem', init=TRUE['apothem'], refined=True,
                    bounds=(TRUE['apothem']-5.0, TRUE['apothem']+5.0)))

# Calibrate (the autocalibrate path picks up the spec's lattice mode).
# Note: with synthetic noiseless rings the strain falls to <1 µε in a few iters.
print('Spec configured with lattice=', spec.lattice, ' and Apothem refinable.')

## 4. `LatticeOrientation` for rotated hex axes

If the detector's hex columns aren't axis-aligned with the lab Y direction (which happens often — physical mounting variance), set `LatticeOrientation` (in degrees) in the spec. It rotates the entire lattice mapping before tilts are applied, and is differentiable so LM can refine it.

In [ ]:
# Add a refinable orientation parameter (small expected range).
spec.add(Parameter('LatticeOrientation', init=0.0, refined=True,
                    bounds=(-5.0, 5.0)))
print('LatticeOrientation now refinable in [-5, 5] degrees.')

## Caveats / when to use

* Only switch to `hex_offset_y` if you actually have a hex-pixel detector. Cartesian PE/GE/Varex with hex mode set will give wrong rings.
* `Apothem` replaces `pxY` / `pxZ` as the geometric scale; the spec's `pxY` / `pxZ` fields are then *derived* (2a, a√3), not refined directly.
* `LatticeOrientation` is *intentional* — only refine it if you suspect physical detector misalignment. Otherwise pin at 0 to remove a degree of freedom.